In [ ]:
import pandas as pd
import numpy as np

In [ ]:
!python -c "import numpy; print(numpy.__version__)"

In [ ]:
#!pip install "numpy<2" --upgrade

In [ ]:
# Select your NDD and the date
ndd = 'FTD'

In [ ]:
# Select the NDD case files created in step 01


#cases= pd.read_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/NDD_rule_of_two/DEM_cases_n2066_rule_of_two.csv')
#cases = pd.read_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/NDD_rule_of_two/AD_cases_n572_rule_of_two.csv')
#cases = pd.read_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/NDD_rule_of_two/PD_cases_n1553_rule_of_two.csv')
#cases = pd.read_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/NDD_rule_of_two/ALS_cases_n122_rule_of_two.csv')
#cases = pd.read_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/NDD_rule_of_two/VAS_cases_n315_rule_of_two.csv')
cases = pd.read_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/NDD_rule_of_two/FTD_cases_n149_rule_of_two.csv')




cases

In [ ]:
# Rename columns to match my code
cases = cases.rename(columns = {'person_id':'ID', f'{ndd}_date':f'{ndd}_DATE'}) 
cases = cases[['ID', 'date_of_birth', 'sex_at_birth', f'{ndd}_DATE']]
cases = cases.sort_values(by = f'{ndd}_DATE')
cases

In [ ]:
# Combine cases and controls
df = pd.concat([cases])

#Check to make sure no duplicate IDs
print(df.ID.value_counts())

df = df.sort_values(by = f'{ndd}_DATE')
df = df.drop_duplicates(subset = 'ID', keep = 'first')

#Check to make sure no duplicate IDs
print(df.ID.value_counts())

df

In [ ]:
# Add DEATH YEAR from file created above
d = pd.read_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/other/death_date.csv')
d = d.rename(columns = {'person_id':'ID', 'death_date':'DATE_OF_DEATH'})
d

In [ ]:
#Merge with cases/controls
df = cases.merge(d, left_on = 'ID', right_on = 'ID', how = 'left')
df

In [ ]:
# eliminate people who were diagnosed at death
df = df[df[f'{ndd}_DATE'] != df['DATE_OF_DEATH']]
df

In [ ]:
# Add recruit year from file created in step 02
r = pd.read_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/other/primary_consent.csv')
r = r.rename(columns = {'person_id':'ID', 'primary_consent_date':'recruit_date'})
r

In [ ]:
#Merge with cases/controls
df = df.merge(r, left_on = 'ID', right_on = 'ID', how = 'left')
df

In [ ]:
cases = df

# Add calculated fields

In [ ]:
#Calculate the tenure, i.e. the time from the beginning of the study to their time of NDD diagnosis
cases['tenure'] = (pd.to_datetime(cases[ndd + '_DATE'], errors = 'coerce') - pd.to_datetime(cases['recruit_date'])).dt.days/365

#Add age_at_tenure to people with an NDD
cases['age_at_tenure'] = (pd.to_datetime(cases[ndd + '_DATE'], errors = 'coerce') - pd.to_datetime(cases['date_of_birth'])).dt.days/365

In [ ]:
cases

In [ ]:
# eliminate people who were diagnosed at death
cases = cases[cases[f'{ndd}_DATE'] != cases['DATE_OF_DEATH']]
cases

In [ ]:
# removed people who received NDD diagnosis before they joined AoU
cases = cases[cases['tenure'] > 0]
cases

In [ ]:
#Remove cases without TOWNEND 
cases = cases[~cases['sex_at_birth'].isna()]
cases = cases[~cases['date_of_birth'].isna()]
cases = cases[~cases[f'{ndd}_DATE'].isna()]
cases

In [ ]:
#START_DATE = '1999-01-01'
START_DATE = '2015-01-01'
STUDY_ENDS = '2024-01-01'

In [ ]:
#Calculate the tenure, i.e. the time from the beginning of the study to their time of NDD diagnosis
cases['tenure'] = (pd.to_datetime(cases[ndd + '_DATE'], errors = 'coerce') - pd.to_datetime(START_DATE)).dt.days/365
cases['tenure_date'] = cases[ndd + '_DATE']

cases

In [ ]:
# removed early onset
cases = cases[cases['age_at_tenure'] >= 60]
cases

In [ ]:
print(len(cases))
cases = cases[cases['tenure'] >= 5]
print(len(cases))
cases

In [ ]:
cases.to_csv(f'/home/jupyter/workspace/WORKSPACE_BUCKET/data/NDD_rule_of_two/{ndd}_with_tenure_JUNE_25_2026.csv', header = True, index = None)